# Synthetic Dataset Repair, Redistribution, and Regeneration Policy

This notebook is for **repairing** the current 20-term synthetic dataset after LLM-judge review.

It helps you:
1. Load the current dataset and judge outputs
2. Inspect label/split/term distributions
3. Identify exact duplicates and other deletion candidates
4. Simulate cleanup before deleting anything
5. Quantify the gaps created by cleanup
6. Propose a regeneration policy per term
7. Export clean planning artifacts for the next generation pass

This notebook does **not** replace the judge notebook. It is the follow-up workflow for dataset repair and policy design.

In [10]:
import json
import csv
import difflib
from pathlib import Path
from collections import Counter, defaultdict

REPO_ROOT = Path("../..").resolve()
DATA_DIR = REPO_ROOT / "data" / "singlish" / "synthetic" / "20-terms"
GROUND_TRUTH_FILE = DATA_DIR / "chosen-20-terms.json"
JUDGE_RESULTS_FILE = DATA_DIR / "judge_results.jsonl"
REPAIR_DIR = DATA_DIR / "repair_outputs"
REPAIR_DIR.mkdir(parents=True, exist_ok=True)

with open(GROUND_TRUTH_FILE, encoding="utf-8") as f:
    ground_truth_list = json.load(f)
TERM_LOOKUP = {entry["term"].casefold(): entry for entry in ground_truth_list}

print("Data directory:", DATA_DIR)
print("Repair outputs:", REPAIR_DIR)
print("Terms loaded:", len(TERM_LOOKUP))

Data directory: /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms
Repair outputs: /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs
Terms loaded: 20


## Load current dataset and judge outputs
Build a row-level table with stable IDs so cleanup decisions can be tracked explicitly.

In [11]:
all_rows = []
for split_name in ["train", "val", "test"]:
    split_file = DATA_DIR / f"{split_name}.jsonl"
    with open(split_file, encoding="utf-8") as f:
        for split_index, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row["split"] = split_name
            row["split_index"] = split_index
            row["row_id"] = f"{split_name}:{split_index}"
            all_rows.append(row)

judge_by_row_id = {}
if JUDGE_RESULTS_FILE.exists():
    with open(JUDGE_RESULTS_FILE, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            result = json.loads(line)
            judge_by_row_id[f"{result['split']}:{result['row_index']}"] = result

for row in all_rows:
    judge = judge_by_row_id.get(row["row_id"])
    row["judge_loaded"] = judge is not None
    if judge:
        row["judge_label"] = judge.get("judge_label")
        row["judge_confidence"] = judge.get("judge_confidence")
        row["judge_target_group"] = judge.get("judge_target_group")
        row["singlish_realism"] = judge.get("singlish_realism")
        row["agreement"] = judge.get("agreement")
        row["reasoning"] = judge.get("reasoning")

print(f"Loaded dataset rows: {len(all_rows)}")
print(f"Judge results matched: {sum(1 for r in all_rows if r['judge_loaded'])}")
print(f"Labels: {Counter(r['label'] for r in all_rows)}")
print(f"Splits: {Counter(r['split'] for r in all_rows)}")

Loaded dataset rows: 540
Judge results matched: 180
Labels: Counter({'hate': 180, 'normal': 180, 'offensive': 180})
Splits: Counter({'train': 180, 'val': 180, 'test': 180})


## Current distribution snapshot
This is the baseline before any deletions.

In [12]:
print("GLOBAL LABEL DISTRIBUTION")
print(Counter(r["label"] for r in all_rows))

print("\nLABEL DISTRIBUTION BY SPLIT")
by_split_label = defaultdict(Counter)
for row in all_rows:
    by_split_label[row["split"]][row["label"]] += 1
for split_name in ["train", "val", "test"]:
    print(split_name, dict(by_split_label[split_name]))

print("\nROWS PER TERM")
by_term = Counter(r["source_term"] for r in all_rows)
for term, count in sorted(by_term.items()):
    print(f"  {term:>30}: {count}")

GLOBAL LABEL DISTRIBUTION
Counter({'hate': 180, 'normal': 180, 'offensive': 180})

LABEL DISTRIBUTION BY SPLIT
train {'hate': 60, 'normal': 60, 'offensive': 60}
val {'hate': 60, 'offensive': 60, 'normal': 60}
test {'hate': 60, 'offensive': 60, 'normal': 60}

ROWS PER TERM
                          CB kia: 27
                       Zhu Bajie: 27
                          ah kua: 27
                           anneh: 27
                     apu neh neh: 27
                        big fuck: 27
                       blur cock: 27
                       bo lan pa: 27
                      cheeko pek: 27
                       cina babi: 27
                   fatty bom bom: 27
                          goondu: 27
                        jiak sai: 27
                          keling: 27
                    lan jiao bin: 27
        nothing but a prostitute: 27
                         pukimak: 27
                           rabak: 27
                    siao cha bor: 27
                       s

## Detect deletion candidates
This combines exact duplicates, cross-split duplicate leakage, and optionally high-confidence judge disagreements.

In [13]:
REMOVE_HIGH_CONFIDENCE_DISAGREEMENTS = False
REMOVE_LOW_REALISM = False

exact_duplicate_groups = defaultdict(list)
for row in all_rows:
    exact_duplicate_groups[(row["label"], row["text"])].append(row)

exact_duplicate_candidates = []
for (_, _), rows_in_group in exact_duplicate_groups.items():
    if len(rows_in_group) > 1:
        # Keep first occurrence, mark the rest as deletion candidates
        for duplicate_row in rows_in_group[1:]:
            exact_duplicate_candidates.append(
                {
                    "row_id": duplicate_row["row_id"],
                    "reason": "exact_duplicate",
                    "split": duplicate_row["split"],
                    "label": duplicate_row["label"],
                    "source_term": duplicate_row["source_term"],
                    "text": duplicate_row["text"],
                }
            )

cross_split_text_groups = defaultdict(list)
for row in all_rows:
    cross_split_text_groups[row["text"]].append(row)

cross_split_duplicate_candidates = []
for text, rows_in_group in cross_split_text_groups.items():
    splits = {row["split"] for row in rows_in_group}
    if len(splits) > 1:
        keep_row = sorted(rows_in_group, key=lambda r: (r["split"], r["split_index"]))[0]
        for row in rows_in_group:
            if row["row_id"] != keep_row["row_id"]:
                cross_split_duplicate_candidates.append(
                    {
                        "row_id": row["row_id"],
                        "reason": "cross_split_duplicate_text",
                        "split": row["split"],
                        "label": row["label"],
                        "source_term": row["source_term"],
                        "text": row["text"],
                    }
                )

judge_candidates = []
for row in all_rows:
    if REMOVE_HIGH_CONFIDENCE_DISAGREEMENTS and row.get("judge_loaded") and row.get("judge_confidence") == "high" and not row.get("agreement", True):
        judge_candidates.append(
            {
                "row_id": row["row_id"],
                "reason": "high_confidence_judge_disagreement",
                "split": row["split"],
                "label": row["label"],
                "source_term": row["source_term"],
                "text": row["text"],
            }
        )
    if REMOVE_LOW_REALISM and row.get("singlish_realism") == "low":
        judge_candidates.append(
            {
                "row_id": row["row_id"],
                "reason": "low_realism",
                "split": row["split"],
                "label": row["label"],
                "source_term": row["source_term"],
                "text": row["text"],
            }
        )

candidate_map = {}
for candidate in exact_duplicate_candidates + cross_split_duplicate_candidates + judge_candidates:
    candidate_map[candidate["row_id"]] = candidate

delete_candidates = list(candidate_map.values())

print("Deletion candidate summary")
print("  exact duplicates:", len(exact_duplicate_candidates))
print("  cross-split duplicate text:", len(cross_split_duplicate_candidates))
print("  judge-driven candidates:", len(judge_candidates))
print("  unique rows flagged:", len(delete_candidates))

for candidate in delete_candidates[:20]:
    print(f"\n[{candidate['reason']}] {candidate['row_id']} | {candidate['label']} | {candidate['source_term']}")
    print(candidate['text'][:180])

Deletion candidate summary
  exact duplicates: 110
  cross-split duplicate text: 106
  judge-driven candidates: 0
  unique rows flagged: 149

[exact_duplicate] train:7 | offensive | ah kua
Stop acting like an ah kua and man up.

[cross_split_duplicate_text] val:24 | normal | apu neh neh
The term apu neh neh is a derogatory slur that should never be used in polite conversation.

[cross_split_duplicate_text] val:25 | normal | apu neh neh
The term apu neh neh is a derogatory slur that should never be used in polite conversation.

[exact_duplicate] test:26 | normal | apu neh neh
The term apu neh neh is a derogatory slur that should not be used in polite conversation.

[cross_split_duplicate_text] val:21 | offensive | apu neh neh
The term apu neh neh is a derogatory slur used in Singapore and Malaysia.

[cross_split_duplicate_text] val:23 | offensive | apu neh neh
The term apu neh neh is a derogatory slur used in Singapore and Malaysia.

[exact_duplicate] test:21 | offensive | apu neh neh
T

## Simulate cleanup and inspect the post-cleanup distribution
Nothing is deleted here yet. This only shows the effect if the flagged rows were removed.

In [14]:
delete_row_ids = {candidate['row_id'] for candidate in delete_candidates}
cleaned_rows = [row for row in all_rows if row['row_id'] not in delete_row_ids]

print("Before cleanup:", len(all_rows))
print("After cleanup: ", len(cleaned_rows))
print("Rows removed:  ", len(all_rows) - len(cleaned_rows))

print("\nPOST-CLEANUP LABEL DISTRIBUTION")
print(Counter(r['label'] for r in cleaned_rows))

print("\nPOST-CLEANUP LABEL DISTRIBUTION BY SPLIT")
post_by_split_label = defaultdict(Counter)
for row in cleaned_rows:
    post_by_split_label[row['split']][row['label']] += 1
for split_name in ['train', 'val', 'test']:
    print(split_name, dict(post_by_split_label[split_name]))

print("\nLABEL DELTAS")
pre_labels = Counter(r['label'] for r in all_rows)
post_labels = Counter(r['label'] for r in cleaned_rows)
for label in ['hate', 'offensive', 'normal']:
    print(f"  {label:>10}: before={pre_labels[label]:>3} after={post_labels[label]:>3} delta={post_labels[label]-pre_labels[label]:>4}")

Before cleanup: 540
After cleanup:  391
Rows removed:   149

POST-CLEANUP LABEL DISTRIBUTION
Counter({'hate': 157, 'normal': 133, 'offensive': 101})

POST-CLEANUP LABEL DISTRIBUTION BY SPLIT
train {'hate': 53, 'normal': 46, 'offensive': 35}
val {'hate': 53, 'offensive': 35, 'normal': 42}
test {'hate': 51, 'offensive': 31, 'normal': 45}

LABEL DELTAS
        hate: before=180 after=157 delta= -23
   offensive: before=180 after=101 delta= -79
      normal: before=180 after=133 delta= -47


## Audit term overload and generation realism
This gives a first-pass policy suggestion for which terms are flexible versus hate-dominant.

In [15]:
def infer_term_policy(entry):
    text = ((entry.get('definitions') or '') + '\n' + (entry.get('usage_notes') or '')).lower()
    severe_markers = ['racist', 'slur', 'extremely offensive', 'ethnic slur', 'should not be used']
    offensive_markers = ['derogatory', 'offensive', 'vulgar', 'contempt']

    severe_hits = sum(marker in text for marker in severe_markers)
    offensive_hits = sum(marker in text for marker in offensive_markers)

    if severe_hits >= 2:
        policy = 'hate_only_candidate'
        rationale = 'definition/usage notes strongly indicate slur-like or highly loaded hate usage'
    elif severe_hits >= 1 or offensive_hits >= 3:
        policy = 'hate_dominant'
        rationale = 'term appears heavily loaded or protected-group-linked, but may allow limited non-hate contextual use'
    elif offensive_hits >= 1:
        policy = 'offensive_dominant'
        rationale = 'term is mainly insulting/vulgar and should be usable beyond hate'
    else:
        policy = 'flexible'
        rationale = 'term appears usable across multiple labels with realistic contextual variation'

    return policy, rationale

term_policy_rows = []
for entry in ground_truth_list:
    policy, rationale = infer_term_policy(entry)
    term_policy_rows.append(
        {
            'term': entry['term'],
            'policy': policy,
            'rationale': rationale,
            'categories': ', '.join(entry.get('categories') or []),
        }
    )

for row in term_policy_rows:
    print(f"{row['term']:>30} | {row['policy']:<20} | {row['categories']}")
    print(f"  {row['rationale']}")

                        ah kua | offensive_dominant   | insults
  term is mainly insulting/vulgar and should be usable beyond hate
                         anneh | flexible             | insults
  term appears usable across multiple labels with realistic contextual variation
                   apu neh neh | hate_only_candidate  | insults
  definition/usage notes strongly indicate slur-like or highly loaded hate usage
                      big fuck | offensive_dominant   | insults, vulgar
  term is mainly insulting/vulgar and should be usable beyond hate
                     blur cock | offensive_dominant   | insults, vulgar
  term is mainly insulting/vulgar and should be usable beyond hate
                     bo lan pa | offensive_dominant   | insults, vulgar
  term is mainly insulting/vulgar and should be usable beyond hate
                        CB kia | offensive_dominant   | insults, vulgar
  term is mainly insulting/vulgar and should be usable beyond hate
                    che

## Plan the regeneration gaps
Quantify how many rows are now missing after cleanup so the next generation pass can target only what is needed.

In [16]:
TARGET_LABEL_COUNTS = Counter(r['label'] for r in all_rows)
post_label_counts = Counter(r['label'] for r in cleaned_rows)
label_gap = {label: TARGET_LABEL_COUNTS[label] - post_label_counts[label] for label in TARGET_LABEL_COUNTS}

print('Global label gaps after cleanup:')
for label in ['hate', 'offensive', 'normal']:
    print(f"  {label:>10}: need {label_gap.get(label, 0)}")

term_label_post = defaultdict(Counter)
for row in cleaned_rows:
    term_label_post[row['source_term']][row['label']] += 1

print('\nTerm-by-label post-cleanup snapshot:')
for term in sorted(term_label_post):
    counts = term_label_post[term]
    print(f"  {term:>30}: hate={counts['hate']}, offensive={counts['offensive']}, normal={counts['normal']}")

policy_by_term = {row['term']: row['policy'] for row in term_policy_rows}
print('\nSuggested regeneration strategy:')
for term in sorted(policy_by_term):
    policy = policy_by_term[term]
    counts = term_label_post[term]
    if policy in {'hate_only_candidate', 'hate_dominant'}:
        suggestion = 'Prioritise hate replenishment; allow generic Singlish non-hate rows if exact term usage is unrealistic.'
    elif policy == 'offensive_dominant':
        suggestion = 'Replenish offensive first, then normal; keep hate only where protected-group targeting is explicit.'
    else:
        suggestion = 'Safe to regenerate across all labels with term-conditioned prompts.'
    print(f"\n  {term} [{policy}]")
    print(f"    current counts: hate={counts['hate']}, offensive={counts['offensive']}, normal={counts['normal']}")
    print(f"    strategy: {suggestion}")

original_split_label_counts = defaultdict(Counter)
for row in all_rows:
    original_split_label_counts[row['split']][row['label']] += 1

cleaned_split_label_counts = defaultdict(Counter)
for row in cleaned_rows:
    cleaned_split_label_counts[row['split']][row['label']] += 1

split_label_gap = defaultdict(dict)
for split_name in ['train', 'val', 'test']:
    for label in ['hate', 'offensive', 'normal']:
        split_label_gap[split_name][label] = (
            original_split_label_counts[split_name][label] - cleaned_split_label_counts[split_name][label]
        )

original_hate_target_counts = defaultdict(Counter)
for row in all_rows:
    if row['label'] == 'hate':
        original_hate_target_counts[row['split']][row.get('target_group', 'none')] += 1

cleaned_hate_target_counts = defaultdict(Counter)
for row in cleaned_rows:
    if row['label'] == 'hate':
        cleaned_hate_target_counts[row['split']][row.get('target_group', 'none')] += 1

hate_target_gap_by_split = defaultdict(dict)
for split_name in ['train', 'val', 'test']:
    groups = set(original_hate_target_counts[split_name]) | set(cleaned_hate_target_counts[split_name])
    for target_group in sorted(groups):
        gap = original_hate_target_counts[split_name][target_group] - cleaned_hate_target_counts[split_name][target_group]
        if gap > 0:
            hate_target_gap_by_split[split_name][target_group] = gap

removed_hate_blueprints = []
for row in all_rows:
    if row['row_id'] in delete_row_ids and row['label'] == 'hate':
        removed_hate_blueprints.append(
            {
                'row_id': row['row_id'],
                'split': row['split'],
                'source_term': row['source_term'],
                'target_group': row.get('target_group', 'none'),
                'source_category': row.get('source_category', ''),
                'text': row['text'],
            }
        )

print('\nSplit-level gaps:')
for split_name in ['train', 'val', 'test']:
    print(split_name, split_label_gap[split_name])

print('\nHate target-group gaps by split:')
for split_name in ['train', 'val', 'test']:
    print(split_name, hate_target_gap_by_split[split_name])

print('\nRemoved hate blueprints:', len(removed_hate_blueprints))

Global label gaps after cleanup:
        hate: need 23
   offensive: need 79
      normal: need 47

Term-by-label post-cleanup snapshot:
                          CB kia: hate=6, offensive=2, normal=9
                       Zhu Bajie: hate=9, offensive=7, normal=8
                          ah kua: hate=9, offensive=7, normal=9
                           anneh: hate=9, offensive=9, normal=9
                     apu neh neh: hate=9, offensive=5, normal=5
                        big fuck: hate=6, offensive=5, normal=6
                       blur cock: hate=9, offensive=2, normal=5
                       bo lan pa: hate=5, offensive=2, normal=1
                      cheeko pek: hate=9, offensive=8, normal=9
                       cina babi: hate=9, offensive=7, normal=9
                   fatty bom bom: hate=7, offensive=0, normal=5
                          goondu: hate=9, offensive=8, normal=4
                        jiak sai: hate=9, offensive=3, normal=7
                          kelin

## Export repair planning artifacts
This writes the delete list and the inferred term policy so you can use them in the next generation step.

In [17]:
delete_candidates_file = REPAIR_DIR / 'delete_candidates.json'
term_policy_file = REPAIR_DIR / 'term_policy.json'
post_cleanup_summary_file = REPAIR_DIR / 'post_cleanup_summary.json'
removed_hate_blueprints_file = REPAIR_DIR / 'removed_hate_blueprints.json'

with open(delete_candidates_file, 'w', encoding='utf-8') as f:
    json.dump(delete_candidates, f, ensure_ascii=False, indent=2)

with open(term_policy_file, 'w', encoding='utf-8') as f:
    json.dump(term_policy_rows, f, ensure_ascii=False, indent=2)

with open(removed_hate_blueprints_file, 'w', encoding='utf-8') as f:
    json.dump(removed_hate_blueprints, f, ensure_ascii=False, indent=2)

post_cleanup_summary = {
    'before_total': len(all_rows),
    'after_total': len(cleaned_rows),
    'rows_removed': len(all_rows) - len(cleaned_rows),
    'pre_label_counts': Counter(r['label'] for r in all_rows),
    'post_label_counts': Counter(r['label'] for r in cleaned_rows),
    'label_gap': label_gap,
    'split_label_gap': {split_name: dict(split_label_gap[split_name]) for split_name in split_label_gap},
    'hate_target_gap_by_split': {split_name: dict(hate_target_gap_by_split[split_name]) for split_name in hate_target_gap_by_split},
}
with open(post_cleanup_summary_file, 'w', encoding='utf-8') as f:
    json.dump(post_cleanup_summary, f, ensure_ascii=False, indent=2)

print('Wrote:', delete_candidates_file)
print('Wrote:', term_policy_file)
print('Wrote:', removed_hate_blueprints_file)
print('Wrote:', post_cleanup_summary_file)

Wrote: /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/delete_candidates.json
Wrote: /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/term_policy.json
Wrote: /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/removed_hate_blueprints.json
Wrote: /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/post_cleanup_summary.json


## Write cleaned dataset files
This writes the post-cleanup dataset into new split files so you can use them as the basis for regeneration and re-evaluation.

In [18]:
CLEANED_SPLIT_DIR = REPAIR_DIR / 'cleaned_splits'
CLEANED_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

rows_by_split = defaultdict(list)
for row in cleaned_rows:
    export_row = {
        'text': row['text'],
        'label': row['label'],
        'target_group': row.get('target_group', 'none'),
        'source_term': row['source_term'],
        'source_category': row.get('source_category', ''),
    }
    rows_by_split[row['split']].append(export_row)

cleaned_paths = {}
for split_name in ['train', 'val', 'test']:
    output_path = CLEANED_SPLIT_DIR / f'{split_name}.jsonl'
    cleaned_paths[split_name] = output_path
    with open(output_path, 'w', encoding='utf-8') as f:
        for row in rows_by_split[split_name]:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(f'Wrote {len(rows_by_split[split_name])} rows -> {output_path}')

combined_cleaned_path = CLEANED_SPLIT_DIR / 'all_cleaned.jsonl'
with open(combined_cleaned_path, 'w', encoding='utf-8') as f:
    for split_name in ['train', 'val', 'test']:
        for row in rows_by_split[split_name]:
            out = dict(row)
            out['split'] = split_name
            f.write(json.dumps(out, ensure_ascii=False) + '\n')
print(f'Wrote combined cleaned dataset -> {combined_cleaned_path}')

cleaned_manifest = {
    'cleaned_split_dir': str(CLEANED_SPLIT_DIR),
    'combined_cleaned_path': str(combined_cleaned_path),
    'split_paths': {k: str(v) for k, v in cleaned_paths.items()},
    'counts_by_split': {k: len(v) for k, v in rows_by_split.items()},
}
cleaned_manifest_file = REPAIR_DIR / 'cleaned_manifest.json'
with open(cleaned_manifest_file, 'w', encoding='utf-8') as f:
    json.dump(cleaned_manifest, f, ensure_ascii=False, indent=2)
print(f'Wrote manifest -> {cleaned_manifest_file}')

Wrote 134 rows -> /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/cleaned_splits/train.jsonl
Wrote 130 rows -> /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/cleaned_splits/val.jsonl
Wrote 127 rows -> /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/cleaned_splits/test.jsonl
Wrote combined cleaned dataset -> /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/cleaned_splits/all_cleaned.jsonl
Wrote manifest -> /Users/shariff/Documents/github/11th-DAP-hate-speech/data/singlish/synthetic/20-terms/repair_outputs/cleaned_manifest.json
